## RAG (Retrieval Augmented Generation) based on a dataset of 20,000 scraped Amazon products

#### For our 2nd agent, we will be asking OpenAI to estimate the price of one of our deals.

In [1]:
import os
import logging
from dotenv import load_dotenv
from huggingface_hub import login
import numpy as np
import re
from sentence_transformers import SentenceTransformer
import chromadb
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from litellm import completion
from tqdm.notebook import tqdm
from agents.evaluator import evaluate
from agents.items import Item

c:\Users\nraja\OneDrive\Desktop\llama\llm_engineering\.venv\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
load_dotenv(override=True)
DB = "products_vectorstore"

In [3]:
hf_token = os.environ['HF_TOKEN']
login(token=hf_token, add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
LITE_MODE = True

In [5]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)
print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

Loaded 20,000 training items, 1,000 validation items, 1,000 test items


## Create a Chroma Datastore

In [6]:
client = chromadb.PersistentClient(path=DB)

## SentenceTransformer

all-MiniLM-L6-v2 is a Sentence Transformer model that converts text into 384-dimensional embeddings.

all-MiniLM-L6-v2 - it converts text into vectors

Computers cannot easily compare text:

"What is AI?"

and

"Explain Artificial Intelligence"

But after converting them to vectors:

Vector A

Vector B

the computer can calculate similarity.

In [7]:
encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [8]:
vector = encoder.encode(["Hyderabad is the fastest growing city in world"])
print(vector.shape)
vector

(1, 384)


array([[ 6.06092587e-02,  1.38203511e-02, -1.07745044e-01,
         3.40634175e-02,  7.51688564e-03, -3.58791873e-02,
         5.22383861e-03, -5.13103865e-02, -1.03477947e-01,
         2.15054452e-02,  9.07874703e-02, -6.30934611e-02,
         2.92765033e-02,  4.62901704e-02, -1.34423014e-03,
        -3.31487171e-02,  4.74809743e-02, -3.34101245e-02,
        -2.31984891e-02, -1.34597510e-01, -8.24984536e-02,
        -5.29910848e-02,  1.05495356e-01, -1.33152297e-02,
         1.40052577e-02,  2.45516356e-02,  1.41249169e-02,
         5.67468107e-02,  4.76884805e-02, -1.23861188e-04,
         6.58254279e-03,  8.40832517e-02,  6.14287965e-02,
         2.63760854e-02, -3.40313055e-02,  1.00713717e-02,
         6.93218363e-03,  1.89832952e-02,  8.11363831e-02,
         4.60372828e-02,  6.00288101e-02, -2.91681644e-02,
        -5.08140437e-02, -5.41745536e-02, -5.61286323e-03,
         2.14485545e-03, -1.38455778e-02,  1.71250980e-02,
        -1.87227726e-02, -1.22031629e-01, -7.74061382e-0

## Populating Chroma Database

Product:

Apple iPhone 15 Pro Max

↓

Sentence Transformer:

[0.21, -0.45, 0.88, ...]

↓

Store in ChromaDB:

ID: doc_1

Summary: Apple iPhone 15 Pro Max

Category: Electronics

Price: 1099

Vector: [0.21, -0.45, ...]

In [9]:
collection_name = "products"
existing_collection_names = [collection.name for collection in client.list_collections()]

if collection_name not in existing_collection_names:
    collection = client.create_collection(collection_name)
    for i in tqdm(range(0, len(train), 200)):
        documents = [item.summary for item in train[i: i+200]]
        vectors = encoder.encode(documents).astype(float).tolist()
        metadatas = [{"category": item.category, "price": item.price} for item in train[i: i+200]]
        ids = [f"doc_{j}" for j in range(i, i+200)]
        ids = ids[:len(documents)]
        collection.add(ids=ids, documents=documents, embeddings=vectors, metadatas=metadatas)

collection = client.get_or_create_collection(collection_name)

## Visualize the vectorized data

Ex :- We will reduce 384 dimensions -> 2 dimensions.

For this we will use T-SNE. TSNE stands for t-distributed Stochastic Neighbor Embedding

In [10]:
MAXIMUM_DATAPOINTS = 1_000

In [11]:
CATEGORIES = ['Appliances', 'Automotive', 'Cell_Phones_and_Accessories', 'Electronics','Musical_Instruments', 'Office_Products', 'Tools_and_Home_Improvement', 'Toys_and_Games']
COLORS = ['cyan', 'blue', 'brown', 'orange', 'yellow', 'green' , 'purple', 'red']

In [28]:
result = collection.get(include=['embeddings', 'documents', 'metadatas'], limit=MAXIMUM_DATAPOINTS)
vectors = np.array(result['embeddings'])
documents = result['documents']
categories = [metadata['category'] for metadata in result['metadatas']]
colors = [COLORS[CATEGORIES.index(c)] for c in categories]

In [29]:
# 2D chart
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=4, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='2D Chroma Vectorstore Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# 3D chart
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

In [ ]:
# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=2, color=colors, opacity=0.7),
    text=[f"Category: {c}<br>Text: {d[:50]}..." for c, d in zip(categories, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=1200,
    height=800,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

## We need to give some context to GPT-5.1 by selecting 5 products with similar descriptions

## RAG + GPT

In [13]:
def vector(item):
    return encoder.encode(item.summary)

In [14]:
def find_similars(item):
    vec = vector(item)
    results = collection.query(query_embeddings=vec.astype(float).tolist(), n_results=5)
    documents = results['documents'][0][:]
    prices = [m['price'] for m in results['metadatas'][0][:]]
    return documents, prices

In [15]:
find_similars(test[0])

(['Title: MXR Classic 108 Fuzz Pedal  \nCategory: Guitar Effects Pedals  \nBrand: MXR  \nDescription: A rugged pedal that delivers the classic late‑60s/early‑70s fuzz tone of the original Fuzz Face in a modern, pedal‑board friendly format.  \nDetails: Features volume and fuzz knobs, a buffer switch to prevent oscillation with wah pedals, true‑bypass with LED indicator, battery door for optional AC operation, and a dual‑path cable for flexible routing.',
  'Title: Sony ECM‑VG1 Compact Shotgun Microphone  \nCategory: Audio Equipment  \nBrand: Sony  \nDescription: Lightweight super‑cardioid shotgun microphone delivering clear, natural sound from 40\u202fHz to 20\u202fkHz.  \nDetails: Features an XLR connector, built‑in low‑cut switch, windscreen with internal frame, and a flat frequency response for studio and field recording.',
  'Title: Pocket Talker Ultra System with HED 021 Deluxe Folding Headphone  \nCategory: Hearing Aids  \nBrand: Williams Sound  \nDescription: A portable amplifica

In [16]:
def make_context(similars, prices):
    message = "For context, here are some other items that might be similar to the item you need to estimate.\n\n"
    for similar, price in zip(similars, prices):
        message += f"Potentially related product:\n{similar}\nPrice is ${price:.2f}\n\n"
    return message

In [17]:
documents, prices = find_similars(test[0])
print(make_context(documents, prices))

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Title: MXR Classic 108 Fuzz Pedal  
Category: Guitar Effects Pedals  
Brand: MXR  
Description: A rugged pedal that delivers the classic late‑60s/early‑70s fuzz tone of the original Fuzz Face in a modern, pedal‑board friendly format.  
Details: Features volume and fuzz knobs, a buffer switch to prevent oscillation with wah pedals, true‑bypass with LED indicator, battery door for optional AC operation, and a dual‑path cable for flexible routing.
Price is $169.99

Potentially related product:
Title: Sony ECM‑VG1 Compact Shotgun Microphone  
Category: Audio Equipment  
Brand: Sony  
Description: Lightweight super‑cardioid shotgun microphone delivering clear, natural sound from 40 Hz to 20 kHz.  
Details: Features an XLR connector, built‑in low‑cut switch, windscreen with internal frame, and a flat frequency response for studio and field recording.
Price is $199.00



In [18]:
def messages_for(item, similars, prices):
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}\n\n"
    message += make_context(similars, prices)
    return [{"role": "user", "content": message}]

In [19]:
documents, prices = find_similars(test[0])
print(messages_for(test[0], documents, prices)[0]['content'])

Estimate the price of this product. Respond with the price, no explanation

Title: Excess V2 Distortion/Modulation Pedal  
Category: Music Pedals  
Brand: Old Blood Noise  
Description: A versatile pedal offering distortion and three modulation modes—delay, chorus, and harmonized fifths—with full control over signal routing and expression.  
Details: Features include separate gain, tone, and volume controls; time, depth, and volume per modulation; order switching, soft‑touch bypass, and expression jack for dynamic control.

For context, here are some other items that might be similar to the item you need to estimate.

Potentially related product:
Title: MXR Classic 108 Fuzz Pedal  
Category: Guitar Effects Pedals  
Brand: MXR  
Description: A rugged pedal that delivers the classic late‑60s/early‑70s fuzz tone of the original Fuzz Face in a modern, pedal‑board friendly format.  
Details: Features volume and fuzz knobs, a buffer switch to prevent oscillation with wah pedals, true‑bypass 

In [20]:
def gpt_5__1_rag(item):
    documents, prices = find_similars(item)
    response = completion(model="gpt-5.1", messages=messages_for(item, documents, prices), reasoning_effort="none", seed=42)
    return response.choices[0].message.content

In [22]:
test[0].price

219.0

In [ ]:
gpt_5__1_rag(test[0])

In [ ]:
evaluate(gpt_5__1_rag, test)

In [24]:
import modal
Pricer = modal.Cls.from_name("pricer-service", "Pricer")
pricer = Pricer()

In [25]:
def specialist(item):
    return pricer.price.remote(item.summary)

In [26]:
def get_price(reply):
    reply = reply.replace("$", "").replace(",", "")
    match = re.search(r"[-+]?\d*\.\d+|\d+", reply)
    return float(match.group()) if match else 0

## Deep Neural Network

In [27]:

from agents.deep_neural_network import DeepNeuralNetworkInference

runner = DeepNeuralNetworkInference()
runner.setup()
runner.load("deep_neural_network.pth")

def deep_neural_network(item):
    return runner.inference(item.summary)

In [ ]:
def ensemble(item):
    price1 = get_price(gpt_5__1_rag(item)) # Frontier Agent
    price2 = specialist(item) # Specialist Agent
    price3 = deep_neural_network(item) # Neural Network Agent
    return price1 * 0.8 + price2 * 0.1 + price3 * 0.1


In [ ]:
evaluate(ensemble, test)

In [32]:
root = logging.getLogger()
root.setLevel(logging.INFO)

## Frontier Agent

In [ ]:
from agents.frontier_agent import FrontierAgent

agent = FrontierAgent(collection)
agent.price("Sony ECM-VG1 Compact Shotgun Microphone ")

In [ ]:
agent.price("MXR Classic 108 Fuzz Pedal  ")

## Neural Network Agent

In [ ]:
from agents.neural_network_agent import NeuralNetworkAgent
agent = NeuralNetworkAgent()


In [ ]:
agent.price("MXR Classic 108 Fuzz Pedal  ")

## Ensemble Agent

In [ ]:
from agents.ensemble_agent import EnsembleAgent
agent = EnsembleAgent(collection)

In [ ]:
agent.price("MXR Classic 108 Fuzz Pedal  ")

### Frontier Agent (GPT + RAG), Specialist Agent (fine-tuned Llama), and Neural Network Agent (DNN)